In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image
import json
import os

# 1. Initialize Moondream 2
model_id = "vikhyatk/moondream2"
# Pinning a revision ensures your research results are reproducible
revision = "2025-01-09" 

print("Loading Moondream 2...")
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    revision=revision,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)

# 2. Setup Image and Paths
image_path = r"D:\Y4 Research\datasets\dietary Images\Set1\542.png"
image = Image.open(image_path)
width, height = image.size

# 3. Step 1: Query for claims
print("Extracting claims...")
# Moondream 2 works best when you encode the image once before asking questions
image_embeds = model.encode_image(image)
prompt = "List all health claims on this label. Return as a simple comma-separated list."
answer = model.answer_question(image_embeds, prompt, tokenizer)
claims_list = [c.strip() for c in answer.split(',')]

# 4. Step 2: Detect Bounding Boxes for each claim
structured_results = []

for claim in claims_list:
    print(f"Detecting: {claim}")
    # Moondream 2's .detect() returns normalized coordinates (0-1000)
    result = model.detect(image, claim)
    
    for obj in result.get("objects", []):
        # Scale 0-1000 coordinates to actual pixel values
        x_min = (obj["x_min"] / 1000) * width
        y_min = (obj["y_min"] / 1000) * height
        x_max = (obj["x_max"] / 1000) * width
        y_max = (obj["y_max"] / 1000) * height
        
        structured_results.append({
            "claim_text": claim,
            "box_2d_normalized": [obj["y_min"], obj["x_min"], obj["y_max"], obj["x_max"]],
            "box_2d_pixels": [int(y_min), int(x_min), int(y_max), int(x_max)]
        })

# 5. Save to JSON
output_json = "label_analysis_md2.json"
with open(output_json, "w") as f:
    json.dump(structured_results, f, indent=4)

print(f"Success! Analysis saved to {output_json}")

Loading Moondream 2...


model.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]